# Landslide combined-class step 08: 50-year discounted avoided EAD summary

This notebook applies the same discount-factor method used in the river flooding and coastal flooding analyses to the landslide combined-class EAD outputs.

Assumptions:

- annual EADs are constant over time;
- present value is calculated over `50` years;
- discount rate is `10%`;
- the discount factor is computed as `sum(1 / (1 + r) ** year for year in range(years + 1))`.

This matches the coastal flooding long-timeframe notebook, including **year 0** in the discount-factor sum.

The notebook calculates discounted present values for:

- avoided EAD from protecting existing forests: `Deforestation EAD - Baseline EAD`;
- avoided EAD from reafforestation: `Baseline EAD - Reafforestation EAD`;
- localized increases in damages where those differences are negative at asset level.


In [ ]:
from pathlib import Path

from IPython.display import display
import numpy as np
import pandas as pd


In [ ]:
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
landslide_results_root = base_path / 'dphil_paper_3/results/02_damage_estimates/landslide_damages'

scenario_paths = {
    'minimum': landslide_results_root / 'results_landslide_minimum_scenario_combined_class/damage_estimates',
    'maximum': landslide_results_root / 'results_landslide_maximum_scenario_combined_class/damage_estimates',
}

combined_output_dir = landslide_results_root / 'discounted_50_year_source_and_runout_zones_combined_class'
combined_output_dir.mkdir(parents=True, exist_ok=True)

for scenario_name, damage_estimates_dir in scenario_paths.items():
    if not damage_estimates_dir.exists():
        raise FileNotFoundError(f'Missing damage estimates directory for {scenario_name}: {damage_estimates_dir}')
    required_files = [
        damage_estimates_dir / 'landslide_ead_asset_level_usd_combined_class.csv',
        damage_estimates_dir / 'landslide_ead_sector_summary_usd_combined_class.csv',
        damage_estimates_dir / 'landslide_ead_subsector_summary_usd_combined_class.csv',
        damage_estimates_dir / 'landslide_ead_overall_totals_usd_combined_class.csv',
    ]
    for required_file in required_files:
        if not required_file.exists():
            raise FileNotFoundError(f'Missing required file: {required_file}')

print('Combined output directory:', combined_output_dir)


In [ ]:
discount_years = 50
discount_rate = 0.10
discounted_suffix = 'PV_50Y_10pct'


def compute_discount_factor(years: int, annual_discount_rate: float) -> float:
    return sum(1 / (1 + annual_discount_rate) ** year for year in range(years + 1))


def format_usd_readable(value):
    if pd.isna(value):
        return 'NA'
    value = float(value)
    abs_value = abs(value)
    sign = '-' if value < 0 else ''
    if abs_value >= 1_000_000_000:
        return f'{sign}US${abs_value / 1_000_000_000:,.2f} billion'
    if abs_value >= 1_000_000:
        return f'{sign}US${abs_value / 1_000_000:,.2f} million'
    if abs_value >= 1_000:
        return f'{sign}US${abs_value / 1_000:,.1f} thousand'
    return f'{sign}US${abs_value:,.0f}'


def format_pct(value):
    if pd.isna(value):
        return 'NA'
    return f'{float(value):,.2f}%'


discount_factor_50_years = compute_discount_factor(discount_years, discount_rate)

metadata = pd.DataFrame([
    {
        'Discount_Years': discount_years,
        'Discount_Rate': discount_rate,
        'Discount_Factor': discount_factor_50_years,
        'Method': 'sum(1 / (1 + discount_rate) ** year for year in range(years + 1))',
        'Includes_Year_0': True,
    }
])
metadata_file = combined_output_dir / 'landslide_50_year_discount_factor_10pct_combined_class.csv'
metadata.to_csv(metadata_file, index=False)

print('Discount factor:', discount_factor_50_years)
print('Saved:', metadata_file)
metadata


In [ ]:
benefit_columns = [
    'Protection_Avoided_EAD_USD',
    'Reafforestation_Avoided_EAD_USD',
    'Protection_Increased_Damage_USD',
    'Reafforestation_Increased_Damage_USD',
    'Protection_Gross_Avoided_EAD_USD',
    'Reafforestation_Gross_Avoided_EAD_USD',
]


def add_discounted_columns(table: pd.DataFrame, annual_columns: list[str]) -> pd.DataFrame:
    output_table = table.copy()
    for annual_column in annual_columns:
        if annual_column in output_table.columns:
            output_table[annual_column] = pd.to_numeric(output_table[annual_column], errors='coerce').fillna(0.0)
            output_table[f'{annual_column}_{discounted_suffix}'] = output_table[annual_column] * discount_factor_50_years
            output_table[f'{annual_column}_Readable'] = output_table[annual_column].apply(format_usd_readable)
            output_table[f'{annual_column}_{discounted_suffix}_Readable'] = output_table[f'{annual_column}_{discounted_suffix}'].apply(format_usd_readable)
    return output_table


def add_percent_labels(table: pd.DataFrame) -> pd.DataFrame:
    output_table = table.copy()
    percentage_columns = [column_name for column_name in output_table.columns if column_name.endswith('_Pct') or column_name.endswith('_Share')]
    for percentage_column in percentage_columns:
        output_table[f'{percentage_column}_Label'] = output_table[percentage_column].apply(format_pct)
    return output_table


def build_asset_metrics(asset_ead: pd.DataFrame) -> pd.DataFrame:
    required_columns = [
        'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID',
        'EAD_Baseline_USD', 'EAD_Deforestation_USD', 'EAD_Reafforestation_USD',
    ]
    missing_columns = [required_column for required_column in required_columns if required_column not in asset_ead.columns]
    if missing_columns:
        raise KeyError(f'Asset EAD table missing columns: {missing_columns}')

    asset_metrics = asset_ead[required_columns].copy()
    asset_metrics['Protection_Net_Avoided_EAD_USD'] = asset_metrics['EAD_Deforestation_USD'] - asset_metrics['EAD_Baseline_USD']
    asset_metrics['Reafforestation_Net_Avoided_EAD_USD'] = asset_metrics['EAD_Baseline_USD'] - asset_metrics['EAD_Reafforestation_USD']

    asset_metrics['Protection_Avoided_EAD_USD'] = asset_metrics['Protection_Net_Avoided_EAD_USD']
    asset_metrics['Reafforestation_Avoided_EAD_USD'] = asset_metrics['Reafforestation_Net_Avoided_EAD_USD']

    asset_metrics['Protection_Gross_Avoided_EAD_USD'] = asset_metrics['Protection_Net_Avoided_EAD_USD'].clip(lower=0.0)
    asset_metrics['Reafforestation_Gross_Avoided_EAD_USD'] = asset_metrics['Reafforestation_Net_Avoided_EAD_USD'].clip(lower=0.0)

    asset_metrics['Protection_Increased_Damage_USD'] = (-asset_metrics['Protection_Net_Avoided_EAD_USD']).clip(lower=0.0)
    asset_metrics['Reafforestation_Increased_Damage_USD'] = (-asset_metrics['Reafforestation_Net_Avoided_EAD_USD']).clip(lower=0.0)

    return add_discounted_columns(
        asset_metrics,
        benefit_columns + ['Protection_Net_Avoided_EAD_USD', 'Reafforestation_Net_Avoided_EAD_USD'],
    )


def aggregate_metrics(asset_metrics: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    annual_columns = benefit_columns + [
        'Protection_Net_Avoided_EAD_USD',
        'Reafforestation_Net_Avoided_EAD_USD',
        'EAD_Baseline_USD',
        'EAD_Deforestation_USD',
        'EAD_Reafforestation_USD',
    ]
    summary_table = (
        asset_metrics
        .groupby(group_columns, as_index=False)[annual_columns]
        .sum()
    )
    summary_table['Protection_Avoided_Pct_vs_Deforestation'] = np.where(
        summary_table['EAD_Deforestation_USD'] > 0,
        100.0 * summary_table['Protection_Avoided_EAD_USD'] / summary_table['EAD_Deforestation_USD'],
        np.nan,
    )
    summary_table['Protection_Avoided_Pct_vs_Baseline'] = np.where(
        summary_table['EAD_Baseline_USD'] > 0,
        100.0 * summary_table['Protection_Avoided_EAD_USD'] / summary_table['EAD_Baseline_USD'],
        np.nan,
    )
    summary_table['Reafforestation_Avoided_Pct_vs_Baseline'] = np.where(
        summary_table['EAD_Baseline_USD'] > 0,
        100.0 * summary_table['Reafforestation_Avoided_EAD_USD'] / summary_table['EAD_Baseline_USD'],
        np.nan,
    )

    discounted_columns = benefit_columns + [
        'Protection_Net_Avoided_EAD_USD',
        'Reafforestation_Net_Avoided_EAD_USD',
    ]
    summary_table = add_discounted_columns(summary_table, discounted_columns)
    summary_table = add_percent_labels(summary_table)
    return summary_table


In [ ]:
discounted_tables_by_scenario = {}

for scenario_name, damage_estimates_dir in scenario_paths.items():
    scenario_output_dir = damage_estimates_dir / 'discounted_50_year_source_and_runout_zones_combined_class'
    scenario_output_dir.mkdir(parents=True, exist_ok=True)

    asset_ead = pd.read_csv(damage_estimates_dir / 'landslide_ead_asset_level_usd_combined_class.csv', low_memory=False)
    asset_metrics = build_asset_metrics(asset_ead)

    total_summary = aggregate_metrics(asset_metrics.assign(Group='Total'), ['Group'])
    sector_summary = aggregate_metrics(asset_metrics, ['Sector'])
    subsector_summary = aggregate_metrics(asset_metrics, ['Sector', 'Subsector'])

    for table in [total_summary, sector_summary, subsector_summary]:
        table.insert(0, 'Damage_Case', scenario_name)
        table['Discount_Years'] = discount_years
        table['Discount_Rate'] = discount_rate
        table['Discount_Factor'] = discount_factor_50_years

    asset_metrics.insert(0, 'Damage_Case', scenario_name)
    asset_metrics['Discount_Years'] = discount_years
    asset_metrics['Discount_Rate'] = discount_rate
    asset_metrics['Discount_Factor'] = discount_factor_50_years

    total_file = scenario_output_dir / 'landslide_ead_total_summary_annual_and_50yr_discounted_10pct_combined_class.csv'
    sector_file = scenario_output_dir / 'landslide_ead_sector_summary_annual_and_50yr_discounted_10pct_combined_class.csv'
    subsector_file = scenario_output_dir / 'landslide_ead_subsector_summary_annual_and_50yr_discounted_10pct_combined_class.csv'
    asset_file = scenario_output_dir / 'landslide_ead_asset_level_annual_and_50yr_discounted_10pct_combined_class.csv'

    total_summary.to_csv(total_file, index=False)
    sector_summary.to_csv(sector_file, index=False)
    subsector_summary.to_csv(subsector_file, index=False)
    asset_metrics.to_csv(asset_file, index=False)

    discounted_tables_by_scenario[scenario_name] = {
        'total': total_summary,
        'sector': sector_summary,
        'subsector': subsector_summary,
        'asset': asset_metrics,
    }

    print(f'[{scenario_name}] Saved:', total_file)
    print(f'[{scenario_name}] Saved:', sector_file)
    print(f'[{scenario_name}] Saved:', subsector_file)
    print(f'[{scenario_name}] Saved:', asset_file)


In [ ]:
combined_total = pd.concat(
    [tables['total'] for tables in discounted_tables_by_scenario.values()],
    ignore_index=True,
)
combined_sector = pd.concat(
    [tables['sector'] for tables in discounted_tables_by_scenario.values()],
    ignore_index=True,
)
combined_subsector = pd.concat(
    [tables['subsector'] for tables in discounted_tables_by_scenario.values()],
    ignore_index=True,
)

combined_total_file = combined_output_dir / 'landslide_ead_total_summary_annual_and_50yr_discounted_10pct_min_max_combined_class.csv'
combined_sector_file = combined_output_dir / 'landslide_ead_sector_summary_annual_and_50yr_discounted_10pct_min_max_combined_class.csv'
combined_subsector_file = combined_output_dir / 'landslide_ead_subsector_summary_annual_and_50yr_discounted_10pct_min_max_combined_class.csv'

combined_total.to_csv(combined_total_file, index=False)
combined_sector.to_csv(combined_sector_file, index=False)
combined_subsector.to_csv(combined_subsector_file, index=False)

print('Saved:', combined_total_file)
print('Saved:', combined_sector_file)
print('Saved:', combined_subsector_file)

display(combined_total)


In [ ]:
def min_max_range(table: pd.DataFrame, group_cols: list[str], value_cols: list[str]) -> pd.DataFrame:
    range_rows = []
    for group_key, group in table.groupby(group_cols, dropna=False):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        range_row = dict(zip(group_cols, group_key))
        for value_column in value_cols:
            range_row[f'{value_column}_Min'] = group[value_column].min()
            range_row[f'{value_column}_Max'] = group[value_column].max()
            range_row[f'{value_column}_Min_Readable'] = format_usd_readable(group[value_column].min()) if 'USD' in value_column else format_pct(group[value_column].min())
            range_row[f'{value_column}_Max_Readable'] = format_usd_readable(group[value_column].max()) if 'USD' in value_column else format_pct(group[value_column].max())
        range_rows.append(range_row)
    return pd.DataFrame(range_rows)

range_value_cols = [
    f'Protection_Avoided_EAD_USD_{discounted_suffix}',
    f'Reafforestation_Avoided_EAD_USD_{discounted_suffix}',
    f'Protection_Increased_Damage_USD_{discounted_suffix}',
    f'Reafforestation_Increased_Damage_USD_{discounted_suffix}',
]

sector_range = min_max_range(combined_sector, ['Sector'], range_value_cols)
subsector_range = min_max_range(combined_subsector, ['Sector', 'Subsector'], range_value_cols)
total_range = min_max_range(combined_total, ['Group'], range_value_cols)

sector_range_file = combined_output_dir / 'landslide_ead_sector_50yr_discounted_10pct_ranges_combined_class.csv'
subsector_range_file = combined_output_dir / 'landslide_ead_subsector_50yr_discounted_10pct_ranges_combined_class.csv'
total_range_file = combined_output_dir / 'landslide_ead_total_50yr_discounted_10pct_ranges_combined_class.csv'

sector_range.to_csv(sector_range_file, index=False)
subsector_range.to_csv(subsector_range_file, index=False)
total_range.to_csv(total_range_file, index=False)

print('Saved:', total_range_file)
print('Saved:', sector_range_file)
print('Saved:', subsector_range_file)

display(total_range)
display(sector_range.sort_values(f'Protection_Avoided_EAD_USD_{discounted_suffix}_Max', ascending=False))


In [ ]:
# Compact tables for manuscript drafting.
summary_cols = [
    'Damage_Case',
    'Protection_Avoided_EAD_USD',
    f'Protection_Avoided_EAD_USD_{discounted_suffix}',
    'Protection_Increased_Damage_USD',
    f'Protection_Increased_Damage_USD_{discounted_suffix}',
    'Reafforestation_Avoided_EAD_USD',
    f'Reafforestation_Avoided_EAD_USD_{discounted_suffix}',
    'Reafforestation_Increased_Damage_USD',
    f'Reafforestation_Increased_Damage_USD_{discounted_suffix}',
]

print('National totals:')
display(combined_total[summary_cols])

print('Sector totals:')
display(combined_sector[['Damage_Case', 'Sector'] + summary_cols[1:]].sort_values(['Sector', 'Damage_Case']))

print('Top subsectors by protection PV benefit:')
display(
    combined_subsector[
        ['Damage_Case', 'Sector', 'Subsector'] + summary_cols[1:]
    ]
    .sort_values(f'Protection_Avoided_EAD_USD_{discounted_suffix}', ascending=False)
    .head(20)
)

print('Top subsectors by reafforestation PV benefit:')
display(
    combined_subsector[
        ['Damage_Case', 'Sector', 'Subsector'] + summary_cols[1:]
    ]
    .sort_values(f'Reafforestation_Avoided_EAD_USD_{discounted_suffix}', ascending=False)
    .head(20)
)


In [ ]:
# Generate a short machine-readable drafting table with readable strings.
drafting_rows = []
for table_name, summary_table, label_columns in [
    ('total', combined_total, ['Group']),
    ('sector', combined_sector, ['Sector']),
    ('subsector', combined_subsector, ['Sector', 'Subsector']),
]:
    output_table = summary_table.copy()
    output_table['Table'] = table_name
    for summary_column in summary_cols[1:]:
        output_table[f'{summary_column}_Readable'] = output_table[summary_column].apply(format_usd_readable)
    drafting_rows.append(
        output_table[
            ['Table', 'Damage_Case']
            + label_columns
            + [f'{summary_column}_Readable' for summary_column in summary_cols[1:]]
        ]
    )

drafting_table = pd.concat(drafting_rows, ignore_index=True, sort=False)
drafting_table_file = combined_output_dir / 'landslide_50yr_discounted_10pct_drafting_values_readable_combined_class.csv'
drafting_table.to_csv(drafting_table_file, index=False)
print('Saved:', drafting_table_file)
display(drafting_table.head(30))
